# V2 ConvLSTM - Intra-Cell DEM Features (Paper 5)

Retrains V2 ConvLSTM with **sub-grid topographic heterogeneity features**
computed from the 90m DEM mapped to CHIRPS 0.05 deg cells.

**Paper 4 (original):** `models/base_models_conv_sthymountain_v2.ipynb` with BASIC/KCE/PAFC

**Paper 5 (this notebook):** Same architecture, new feature bundles:
- `BASIC_D10`: BASIC + 10 elevation deciles (p10-p100) per cell
- `BASIC_PCA6`: BASIC + 6 PCA components of intra-cell DEM
- `BASIC_D10_STATS`: BASIC + 10 deciles + 5 statistics (mean, std, skew, kurtosis, range)

**Technique:** Each CHIRPS cell (~5.5 km) contains ~3,477 DEM pixels at 90m.
The deciles capture elevation heterogeneity within each cell.

**GPU:** A100 recommended. Training time ~6-8h per feature bundle.

## 1. Environment Setup and Imports

In [ ]:
# ==================================================
# ENVIRONMENT SETUP AND IMPORTS
# ==================================================

import os, sys

IN_COLAB = 'google.colab' in sys.modules

# Install dependencies in Colab (not available by default)
if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install',
                    'netcdf4', 'h5py', 'xarray', 'h5netcdf', '-q'], check=True)
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
import tensorflow as tf
tf.random.set_seed(42)
import numpy as np
np.random.seed(42)
import pandas as pd
import xarray as xr
from pathlib import Path
import gc, warnings, json, time
from datetime import datetime
from typing import Dict, List, Tuple, Optional
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import clear_output, display

from tensorflow.keras import backend as K
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv2D, ConvLSTM2D, Flatten, Dense, Reshape,
    Lambda, Layer, TimeDistributed, Dropout, BatchNormalization, Add,
    Concatenate, MultiHeadAttention, LayerNormalization, GlobalAveragePooling1D
)
from tensorflow.keras.callbacks import (
    ReduceLROnPlateau, CSVLogger, EarlyStopping, ModelCheckpoint, Callback
)
from tensorflow.keras.optimizers import Adam

# Configure GPU memory growth
try:
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f'GPU memory growth configured for {len(gpus)} GPU(s)')
    else:
        print('No GPU detected - running on CPU')
except RuntimeError as e:
    print(f'GPU configuration warning: {e}')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context('notebook')
warnings.filterwarnings('ignore')
import logging
tf.get_logger().setLevel('ERROR')
logging.getLogger('tensorflow').setLevel(logging.ERROR)

print(f'TensorFlow: {tf.__version__}')
print(f'NumPy: {np.__version__}')
print(f'Colab: {IN_COLAB}')
print('Environment setup complete')

## 2. Configuration (Paper 5 - Intra-Cell DEM)

In [ ]:
# ==================================================
# CONFIGURATION - PAPER 5 INTRA-CELL DEM
# ==================================================

# --- Paper 5 Feature Sets ---
FEATURES_BASIC = [
    'year', 'month', 'month_sin', 'month_cos', 'doy_sin', 'doy_cos',
    'max_daily_precipitation', 'min_daily_precipitation', 'daily_precipitation_std',
    'elevation', 'slope', 'aspect',
]

FEATURES_BASIC_D10 = FEATURES_BASIC + [
    'dem_p10', 'dem_p20', 'dem_p30', 'dem_p40', 'dem_p50',
    'dem_p60', 'dem_p70', 'dem_p80', 'dem_p90', 'dem_p100',
]

FEATURES_BASIC_PCA6 = FEATURES_BASIC + [
    'dem_pca_1', 'dem_pca_2', 'dem_pca_3',
    'dem_pca_4', 'dem_pca_5', 'dem_pca_6',
]

FEATURES_BASIC_D10_STATS = FEATURES_BASIC + [
    'dem_p10', 'dem_p20', 'dem_p30', 'dem_p40', 'dem_p50',
    'dem_p60', 'dem_p70', 'dem_p80', 'dem_p90', 'dem_p100',
    'dem_mean', 'dem_std', 'dem_skewness', 'dem_kurtosis', 'dem_range',
]

# --- Feature scaling groups ---
# Continuous features to scale with StandardScaler
BASE_CONTINUOUS_FEATURES = [
    'year', 'month', 'month_sin', 'month_cos', 'doy_sin', 'doy_cos',
    'max_daily_precipitation', 'min_daily_precipitation', 'daily_precipitation_std',
    'elevation', 'slope', 'aspect',
]
# DEM intra-cell features are also continuous
DEM_CONTINUOUS_FEATURES = [
    'dem_p10', 'dem_p20', 'dem_p30', 'dem_p40', 'dem_p50',
    'dem_p60', 'dem_p70', 'dem_p80', 'dem_p90', 'dem_p100',
    'dem_mean', 'dem_std', 'dem_skewness', 'dem_kurtosis', 'dem_range',
    'dem_pca_1', 'dem_pca_2', 'dem_pca_3', 'dem_pca_4', 'dem_pca_5', 'dem_pca_6',
]

CONFIG = {
    'input_window': 60,
    'horizon': 12,
    'epochs': 150,
    'batch_size': 2,
    'learning_rate': 1e-3,
    'patience': 50,
    'light_mode': False,  # MUST be False for full training
    'light_grid_size': 5,
    'enabled_horizons': [12],
    'loss_weighting': 'uniform',
    'save_window_metrics': False,
    'train_val_split': 0.8,
    'prediction_batch_size': 1,
    'gradient_accumulation_steps': 1,
    # Paper 5 bundles only
    'feature_sets': {
        'BASIC_D10': FEATURES_BASIC_D10,       # 22 features
        'BASIC_PCA6': FEATURES_BASIC_PCA6,      # 18 features
        # 'BASIC_D10_STATS': FEATURES_BASIC_D10_STATS,  # 27 features (uncomment if needed)
    },
}

# Paths
if IN_COLAB:
    CONFIG['base_path'] = Path('/content/drive/MyDrive/ml_precipitation_prediction')
else:
    CONFIG['base_path'] = Path.cwd()
    for p in [CONFIG['base_path'], *CONFIG['base_path'].parents]:
        if (p / '.git').exists():
            CONFIG['base_path'] = p
            break

# Paper 5 extended dataset (with intra-cell DEM features)
CONFIG['data_file'] = CONFIG['base_path'] / 'data' / 'output' / (
    'complete_dataset_extended_dem_features.nc'
)
# Output to _intracell_dem directory
CONFIG['out_root'] = CONFIG['base_path'] / 'models' / 'output' / 'V2_Enhanced_Models_intracell_dem'
CONFIG['out_root'].mkdir(parents=True, exist_ok=True)

print(f'Base path: {CONFIG["base_path"]}')
print(f'Dataset: {CONFIG["data_file"]}')
print(f'Output: {CONFIG["out_root"]}')
print(f'Bundles: {list(CONFIG["feature_sets"].keys())}')
for name, feats in CONFIG['feature_sets'].items():
    print(f'  {name}: {len(feats)} features')

## 3. Data Loading and Validation

In [ ]:
# ==================================================
# DATA LOADING AND VALIDATION
# ==================================================

def load_and_validate_data(config: Dict) -> Tuple[xr.Dataset, int, int]:
    """Load dataset and validate features, applying light mode if enabled."""
    data_file = config['data_file']
    light_mode = config['light_mode']
    light_grid_size = config['light_grid_size']

    print('Loading dataset...')
    if not data_file.exists():
        raise FileNotFoundError(f'Dataset not found: {data_file}')

    ds = xr.open_dataset(data_file)

    if light_mode:
        print(f'*** LIGHT MODE: Using {light_grid_size}x{light_grid_size} subset ***')
        lat_center = len(ds.latitude) // 2
        lon_center = len(ds.longitude) // 2
        lat_start = lat_center - light_grid_size // 2
        lon_start = lon_center - light_grid_size // 2
        ds = ds.isel(
            latitude=slice(lat_start, lat_start + light_grid_size),
            longitude=slice(lon_start, lon_start + light_grid_size)
        )

    n_lat, n_lon = len(ds.latitude), len(ds.longitude)
    print(f'Dataset loaded: time={len(ds.time)}, lat={n_lat}, lon={n_lon}')
    print(f'Variables: {len(ds.data_vars)}')

    # Validate features
    for exp_name, feats in config['feature_sets'].items():
        missing = [f for f in feats if f not in ds.data_vars and f not in ds.coords]
        if missing:
            raise ValueError(f'{exp_name} missing features: {missing}')
        print(f'  {exp_name}: all {len(feats)} features present')

    return ds, n_lat, n_lon

ds, lat, lon = load_and_validate_data(CONFIG)
print('Dataset ready for training')

## 4. Custom Layers and Losses

In [ ]:
# ==================================================
# CUSTOM LAYERS AND LOSSES
# ==================================================

class MultiHorizonLoss(tf.keras.losses.Loss):
    """Weighted loss for multi-horizon forecasting."""
    def __init__(self, horizon_weights=[0.4, 0.35, 0.25], name='multi_horizon_loss', **kwargs):
        super().__init__(name=name, **kwargs)
        self.horizon_weights = tf.constant(horizon_weights, dtype=tf.float32)
    def call(self, y_true, y_pred):
        y_pred = tf.maximum(y_pred, 0.0)
        mse = tf.reduce_mean(tf.square(y_true - y_pred), axis=[2, 3, 4])
        weights = tf.reshape(self.horizon_weights, [1, -1])
        return tf.reduce_mean(tf.reduce_sum(mse * weights, axis=1))
    def get_config(self):
        config = super().get_config()
        config.update({'horizon_weights': self.horizon_weights.numpy().tolist()})
        return config


class CombinedLoss(tf.keras.losses.Loss):
    """Combines multi-horizon MSE with temporal consistency."""
    def __init__(self, horizon_weights=[0.4, 0.35, 0.25], consistency_weight=0.1, name='combined_loss', **kwargs):
        super().__init__(name=name, **kwargs)
        self.horizon_weights = tf.constant(horizon_weights, dtype=tf.float32)
        self.consistency_weight = consistency_weight
    def call(self, y_true, y_pred):
        y_pred = tf.maximum(y_pred, 0.0)
        mse = tf.reduce_mean(tf.square(y_true - y_pred), axis=[2, 3, 4])
        weights = tf.reshape(self.horizon_weights, [1, -1])
        mh_loss = tf.reduce_mean(tf.reduce_sum(mse * weights, axis=1))
        temporal_diffs = tf.abs(y_pred[:, 1:, :, :, :] - y_pred[:, :-1, :, :, :])
        tc_loss = tf.reduce_mean(temporal_diffs)
        return mh_loss + self.consistency_weight * tc_loss
    def get_config(self):
        config = super().get_config()
        config.update({
            'horizon_weights': self.horizon_weights.numpy().tolist(),
            'consistency_weight': self.consistency_weight,
        })
        return config


class SimpleTemporalAttention(Layer):
    """Temporal attention for sequence processing."""
    def __init__(self, units=64, **kwargs):
        super().__init__(**kwargs)
        self.units = units
    def build(self, input_shape):
        self.attention_dense = Dense(1, activation='tanh')
        self.softmax = tf.keras.layers.Softmax(axis=1)
        super().build(input_shape)
    def call(self, inputs):
        scores = self.attention_dense(inputs)
        weights = self.softmax(scores)
        context = tf.reduce_sum(inputs * weights, axis=1)
        return context, weights
    def get_config(self):
        config = super().get_config()
        config.update({'units': self.units})
        return config


class SpatialReshapeLayer(Layer):
    """Reshape: (batch, time, H, W, C) -> (batch, time, H*W*C)."""
    def call(self, inputs):
        batch_size, time_steps, height, width, channels = tf.unstack(tf.shape(inputs))
        return tf.reshape(inputs, [batch_size, time_steps, height * width * channels])
    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[1], input_shape[2] * input_shape[3] * input_shape[4])


class SpatialRestoreLayer(Layer):
    """Restore spatial dimensions after attention."""
    def __init__(self, height, width, channels, **kwargs):
        super().__init__(**kwargs)
        self.height = height
        self.width = width
        self.channels = channels
    def call(self, inputs):
        batch_size, time_steps, _ = tf.unstack(tf.shape(inputs))
        return tf.reshape(inputs, [batch_size, time_steps, self.height, self.width, self.channels])
    def get_config(self):
        config = super().get_config()
        config.update({'height': self.height, 'width': self.width, 'channels': self.channels})
        return config


print('Custom layers and losses defined')

## 5. Model Architecture (ConvLSTM)

In [ ]:
# ==================================================
# MODEL ARCHITECTURE - ConvLSTM (Primary V2 Model)
# ==================================================

def _spatial_head(x, horizon: int, lat: int, lon: int) -> tf.Tensor:
    """Convert ConvLSTM output to (batch, horizon, lat, lon, 1)."""
    if len(x.shape) == 5:
        x = Lambda(lambda t: tf.squeeze(t, axis=1) if tf.shape(t)[1] == 1 else t[:, -1, :, :, :],
                   name='take_last_step')(x)
    x = Conv2D(horizon, (1, 1), padding='same', activation='linear', name='head_conv1x1')(x)
    x = Lambda(lambda t: tf.transpose(t, [0, 3, 1, 2]), name='head_transpose')(x)
    x = Lambda(lambda t: tf.expand_dims(t, -1), name='head_expand_dim')(x)
    return x


def _assert_model_output(model: Model, horizon: int, lat: int, lon: int):
    expected = (None, horizon, lat, lon, 1)
    actual = model.output_shape
    if actual != expected:
        raise ValueError(f'{model.name} output_shape {actual} != expected {expected}')


def build_conv_lstm(n_feats: int, lat: int, lon: int, horizon: int) -> Model:
    """Build ConvLSTM model (primary V2 architecture)."""
    inp = Input(shape=(None, lat, lon, n_feats))
    x = ConvLSTM2D(32, (3, 3), padding='same', return_sequences=True)(inp)
    x = ConvLSTM2D(16, (3, 3), padding='same', return_sequences=False)(x)
    out = _spatial_head(x, horizon, lat, lon)
    model = Model(inp, out, name='ConvLSTM')
    _assert_model_output(model, horizon, lat, lon)
    return model


# Only train ConvLSTM (the primary V2 model for Paper 5)
MODELS = {
    'ConvLSTM': build_conv_lstm,
}

print(f'Models registered: {list(MODELS.keys())}')

## 6. Data Preprocessing

In [ ]:
# ==================================================
# DATA PREPROCESSING
# ==================================================

def windowed_arrays(X, y, input_window, horizon, start_indices=None):
    """Create windowed arrays for sequence-to-sequence learning."""
    seq_X, seq_y = [], []
    T = len(X)
    if T < (input_window + horizon):
        raise ValueError(f'Not enough timesteps ({T}) for windows')
    iterator = start_indices if start_indices is not None else range(T - input_window - horizon + 1)
    for start in iterator:
        if start < 0:
            continue
        end_w = start + input_window
        end_y = end_w + horizon
        if end_y > T:
            continue
        Xw = X[start:end_w]
        yw = y[end_w:end_y]
        if np.isnan(Xw).any() or np.isnan(yw).any():
            continue
        seq_X.append(Xw)
        seq_y.append(yw)
    if not seq_X:
        raise ValueError('No valid windows found')
    return np.asarray(seq_X, dtype=np.float32), np.asarray(seq_y, dtype=np.float32)


def compute_split_indices(total_steps, input_window, horizon, split_ratio):
    """Return train/validation start indices ensuring no temporal leakage."""
    cutoff = max(0, int(total_steps * split_ratio))
    max_start = total_steps - input_window - horizon + 1
    train_end = max(0, cutoff - input_window - horizon + 1)
    train_indices = list(range(0, train_end))
    val_start = min(max_start, max(cutoff, 0))
    val_indices = list(range(val_start, max_start))
    if not train_indices or not val_indices:
        raise ValueError('Split produced empty train/val sets')
    return train_indices, val_indices


def fill_nan_with_median(X, features):
    """Fill NaN in features with per-feature spatial median.

    603 cells (15.2%) have NaN in DEM features because the 90m DEM GeoTIFF
    doesn't cover those cells. Filling with median makes these cells 'average'
    after StandardScaler, minimizing bias.
    """
    nan_count = 0
    for i, feat in enumerate(features):
        if np.isnan(X[..., i]).any():
            # Use median of valid spatial cells (static features: time=0 is representative)
            valid_vals = X[0, :, :, i][~np.isnan(X[0, :, :, i])]
            fill_val = float(np.median(valid_vals)) if len(valid_vals) > 0 else 0.0
            X[..., i] = np.nan_to_num(X[..., i], nan=fill_val)
            nan_count += 1
            print(f'    Filled {feat} NaN with median={fill_val:.1f}')
    if nan_count > 0:
        print(f'    Filled NaN in {nan_count} features')
    return X


def scale_feature_blocks(X_tr, X_va, features):
    """Scale continuous features independently. DEM features scaled as continuous."""
    X_tr_scaled = X_tr.copy()
    X_va_scaled = X_va.copy()

    def apply_scaler(idxs):
        if not idxs:
            return
        scaler = StandardScaler()
        tr_block = X_tr[..., idxs].reshape(-1, len(idxs))
        va_block = X_va[..., idxs].reshape(-1, len(idxs))
        scaler.fit(tr_block)
        X_tr_scaled[..., idxs] = scaler.transform(tr_block).reshape(X_tr[..., idxs].shape)
        X_va_scaled[..., idxs] = scaler.transform(va_block).reshape(X_va[..., idxs].shape)

    # Base continuous features
    cont_indices = [i for i, f in enumerate(features) if f in BASE_CONTINUOUS_FEATURES]
    apply_scaler(cont_indices)

    # DEM intra-cell features (also continuous)
    dem_indices = [i for i, f in enumerate(features) if f in DEM_CONTINUOUS_FEATURES]
    apply_scaler(dem_indices)

    return X_tr_scaled, X_va_scaled


def preprocess_data(ds, config, lat, lon, horizon):
    """Preprocess data per experiment with no leakage."""
    data_splits = {}
    split_ratio = config.get('train_val_split', 0.8)

    for exp_name, features in config['feature_sets'].items():
        print(f'Preprocessing {exp_name}...')
        X = np.stack([ds[feat].values for feat in features], axis=-1)
        y = ds['total_precipitation'].values[..., np.newaxis]

        # Fill NaN in DEM features with per-feature spatial median
        X = fill_nan_with_median(X, features)

        total_steps = X.shape[0]

        train_idx, val_idx = compute_split_indices(
            total_steps, config['input_window'], horizon, split_ratio
        )

        X_tr, y_tr = windowed_arrays(X, y, config['input_window'], horizon, train_idx)
        X_va, y_va = windowed_arrays(X, y, config['input_window'], horizon, val_idx)

        X_tr_s, X_va_s = scale_feature_blocks(X_tr, X_va, features)

        y_scaler = StandardScaler()
        y_tr_s = y_scaler.fit_transform(y_tr.reshape(-1, 1)).reshape(y_tr.shape)
        y_va_s = y_scaler.transform(y_va.reshape(-1, 1)).reshape(y_va.shape)

        data_splits[exp_name] = (
            X_tr_s.astype(np.float32), y_tr_s.astype(np.float32),
            X_va_s.astype(np.float32), y_va_s.astype(np.float32),
            y_scaler
        )
        print(f'  {exp_name}: {X_tr.shape[0]} train, {X_va.shape[0]} val windows')
        print(f'  X shape: {X_tr_s.shape}, y shape: {y_tr_s.shape}')

    return data_splits


print('Preprocessing functions defined')

## 7. Training Infrastructure

In [ ]:
# ==================================================
# TRAINING INFRASTRUCTURE
# ==================================================

def compute_horizon_weights(H, strategy='uniform'):
    if H <= 0:
        return []
    if strategy == 'uniform':
        w = np.ones(H, dtype=np.float32)
    elif strategy == 'linear_decay':
        w = np.linspace(1.0, 0.2, H, dtype=np.float32)
    else:
        w = np.ones(H, dtype=np.float32)
    return (w / w.sum()).tolist()


class TrainingMonitor(Callback):
    def __init__(self, model_name, exp_name):
        super().__init__()
        self.model_name = model_name
        self.exp_name = exp_name
        self.start_time = None
        self.lrs = []
    def on_train_begin(self, logs=None):
        self.start_time = time.time()
    def on_epoch_end(self, epoch, logs=None):
        elapsed = time.time() - self.start_time
        try:
            lr = float(self.model.optimizer.learning_rate)
        except Exception:
            lr = 0.0
        self.lrs.append(lr)
        loss = logs.get('loss', np.nan) if logs else np.nan
        vloss = logs.get('val_loss', np.nan) if logs else np.nan
        print(f'{self.exp_name} - {self.model_name} | Epoch {epoch+1} | '
              f'Loss: {loss:.4f} | Val Loss: {vloss:.4f} | LR: {lr:.2e} | {elapsed:.0f}s')


def batch_predict(model, X, batch_size=1):
    """Predict in mini-batches."""
    import math
    n_samples = X.shape[0]
    num_batches = int(math.ceil(n_samples / float(batch_size)))
    predictions = []
    for b in range(num_batches):
        start = b * batch_size
        end = min(n_samples, start + batch_size)
        batch_pred = model.predict(X[start:end], verbose=0, batch_size=batch_size)
        predictions.append(np.asarray(batch_pred))
    return np.concatenate(predictions, axis=0)


def train_model(model, X_tr, y_tr, X_va, y_va, config, model_name, exp_name, out_dir, horizon):
    """Train a single model with callbacks and save results."""
    metrics_dir = out_dir / f'h{horizon}' / exp_name / 'training_metrics'
    metrics_dir.mkdir(parents=True, exist_ok=True)
    model_path = metrics_dir / f'{model_name}_best_h{horizon}.h5'

    # Save hyperparameters
    hyperparams = {
        'learning_rate': config['learning_rate'],
        'batch_size': config['batch_size'],
        'epochs': config['epochs'],
        'patience': config['patience'],
        'model_params': model.count_params(),
        'version': 'V2_Intracell_DEM',
        'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    }
    with open(metrics_dir / f'{model_name}_hyperparameters.json', 'w') as f:
        json.dump(hyperparams, f, indent=4)

    # Loss selection - CombinedLoss for all Paper 5 bundles (all continuous features)
    horizon_weights = compute_horizon_weights(horizon, config.get('loss_weighting', 'uniform'))
    loss = CombinedLoss(horizon_weights=horizon_weights, consistency_weight=0.1)

    model.compile(optimizer=Adam(config['learning_rate']), loss=loss, metrics=['mae'])
    print(f'Model compiled: {model_name} ({model.count_params():,} params)')
    model.summary()

    # Callbacks
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=config['patience'], restore_best_weights=True, verbose=1),
        ModelCheckpoint(str(model_path), save_best_only=True, monitor='val_loss', verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=config['patience']//2, min_lr=1e-6, verbose=1),
        CSVLogger(str(metrics_dir / f'{model_name}_training_log_h{horizon}.csv'), separator=',', append=False),
        TrainingMonitor(model_name, exp_name),
    ]

    train_batch = max(1, config['batch_size'])
    print(f'Training {model_name} on {exp_name} (batch_size={train_batch})...')

    history = model.fit(
        X_tr, y_tr,
        validation_data=(X_va, y_va),
        epochs=config['epochs'],
        batch_size=train_batch,
        callbacks=callbacks,
        verbose=0
    )

    # Save history
    val_losses = history.history['val_loss']
    best_idx = int(np.argmin(val_losses))
    history_dict = {
        'loss': [float(x) for x in history.history['loss']],
        'val_loss': [float(x) for x in val_losses],
        'mae': [float(x) for x in history.history.get('mae', [])],
        'val_mae': [float(x) for x in history.history.get('val_mae', [])],
        'best_epoch': best_idx,
        'best_val_loss': float(val_losses[best_idx]),
    }
    with open(metrics_dir / f'{model_name}_history.json', 'w') as f:
        json.dump(history_dict, f, indent=4)

    print(f'Training complete. Best val_loss: {val_losses[best_idx]:.4f} at epoch {best_idx+1}')

    # Plot learning curve
    fig, ax = plt.subplots(figsize=(10, 6))
    epochs = range(1, len(history_dict['loss']) + 1)
    ax.plot(epochs, history_dict['loss'], 'b-', label='Train')
    ax.plot(epochs, history_dict['val_loss'], 'r-', label='Val')
    ax.plot(best_idx + 1, val_losses[best_idx], 'r*', markersize=12, label=f'Best: {val_losses[best_idx]:.4f}')
    ax.set_title(f'{model_name} - {exp_name}')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.savefig(metrics_dir / f'{model_name}_learning_curve.png', dpi=150, bbox_inches='tight')
    plt.show()

    return model, history_dict


print('Training functions defined')

## 8. Main Training Loop

In [ ]:
# ==================================================
# MAIN TRAINING LOOP
# ==================================================

results = []
all_histories = {}

for horizon in CONFIG['enabled_horizons']:
    print(f'\n{"="*60}')
    print(f'Training for horizon H={horizon}')
    print(f'{"="*60}')

    data_splits_h = preprocess_data(ds, CONFIG, lat, lon, horizon)

    for exp_name in CONFIG['feature_sets']:
        if exp_name not in data_splits_h:
            print(f'Skipping {exp_name}: data not available')
            continue

        X_tr, y_tr, X_va, y_va, scaler = data_splits_h[exp_name]
        print(f'\n--- {exp_name}: X_tr={X_tr.shape}, y_tr={y_tr.shape} ---')

        for model_name, model_builder in MODELS.items():
            tf.keras.backend.clear_session()
            n_features = len(CONFIG['feature_sets'][exp_name])
            print(f'\n>>> {model_name} with {n_features} features')

            try:
                model = model_builder(n_features, lat, lon, horizon)
                model, history = train_model(
                    model, X_tr, y_tr, X_va, y_va,
                    CONFIG, model_name, exp_name, CONFIG['out_root'], horizon
                )
                all_histories[f'{exp_name}_{model_name}_H{horizon}'] = history

                if not history or not history.get('val_loss'):
                    print(f'Training failed for {model_name}, skipping predictions')
                    continue

                # Generate predictions
                pred_batch = max(1, CONFIG.get('prediction_batch_size', 1))
                y_hat_sc = batch_predict(model, X_va, batch_size=pred_batch)
                if y_hat_sc.shape != y_va.shape:
                    y_hat_sc = y_hat_sc.reshape(y_va.shape)
                y_hat = scaler.inverse_transform(y_hat_sc.reshape(-1, 1)).reshape(y_va.shape)
                y_true = scaler.inverse_transform(y_va.reshape(-1, 1)).reshape(y_va.shape)

                # Per-horizon metrics
                for h in range(horizon):
                    y_true_h = y_true[:, h, ..., 0]
                    y_pred_h = y_hat[:, h, ..., 0]
                    rmse = np.sqrt(mean_squared_error(y_true_h.ravel(), y_pred_h.ravel()))
                    mae = mean_absolute_error(y_true_h.ravel(), y_pred_h.ravel())
                    r2 = r2_score(y_true_h.ravel(), y_pred_h.ravel())
                    results.append({
                        'TotalHorizon': horizon,
                        'Experiment': exp_name,
                        'Model': model_name,
                        'H': h + 1,
                        'RMSE': rmse, 'MAE': mae, 'R^2': r2,
                        'Mean_True_mm': float(np.mean(y_true_h)),
                        'Mean_Pred_mm': float(np.mean(y_pred_h)),
                    })
                    print(f'  H={h+1}: RMSE={rmse:.2f}, MAE={mae:.2f}, R2={r2:.4f}')

                # Compute forecast_dates for ACC (benchmark script 14)
                total_steps = len(ds.time)
                _, val_idx = compute_split_indices(
                    total_steps, CONFIG['input_window'], horizon,
                    CONFIG.get('train_val_split', 0.8)
                )
                times = pd.to_datetime(ds.time.values)
                forecast_dates = []
                for vi in val_idx:
                    sample_dates = []
                    for h_offset in range(horizon):
                        t_idx = vi + CONFIG['input_window'] + h_offset
                        if t_idx < total_steps:
                            sample_dates.append(times[t_idx].strftime('%Y-%m'))
                    if len(sample_dates) == horizon:
                        forecast_dates.append(sample_dates)

                # Export predictions for V10 Late Fusion
                export_dir = CONFIG['out_root'] / 'map_exports' / f'H{horizon}' / exp_name / model_name
                export_dir.mkdir(parents=True, exist_ok=True)
                np.save(export_dir / 'predictions.npy', y_hat.astype(np.float32))
                np.save(export_dir / 'targets.npy', y_true.astype(np.float32))
                metadata = {
                    'model': model_name, 'experiment': exp_name, 'horizon': horizon,
                    'shape': list(y_hat.shape),
                    'rmse_mean': float(np.sqrt(np.mean((y_hat - y_true)**2))),
                    'r2_mean': float(r2_score(y_true.ravel(), y_hat.ravel())),
                    'forecast_dates': forecast_dates,
                    'generated_at': datetime.now().isoformat(),
                }
                with open(export_dir / 'metadata.json', 'w') as f:
                    json.dump(metadata, f, indent=2)
                print(f'  Predictions saved: {export_dir}')
                print(f'  forecast_dates: {len(forecast_dates)} samples x {horizon} horizons')

            except Exception as e:
                print(f'ERROR training {model_name} for {exp_name}: {e}')
                import traceback
                traceback.print_exc()

            tf.keras.backend.clear_session()
            gc.collect()

# Save results
if results:
    res_df = pd.DataFrame(results)
    out_csv = CONFIG['out_root'] / 'metrics_spatial_v2_intracell_dem_h12.csv'
    res_df.to_csv(out_csv, index=False)
    print(f'\nResults saved to {out_csv}')
    print(res_df.groupby(['Experiment', 'Model'])[['RMSE', 'MAE', 'R^2']].mean().round(4))
else:
    print('No results generated')

print('\nTraining complete!')

## 9. Summary and Verification

In [ ]:
# ==================================================
# SUMMARY AND VERIFICATION
# ==================================================

print('=' * 60)
print('V2 ConvLSTM INTRACELL DEM TRAINING COMPLETE')
print('=' * 60)

# Verify output files
for horizon in CONFIG['enabled_horizons']:
    for exp_name in CONFIG['feature_sets']:
        pred_dir = CONFIG['out_root'] / 'map_exports' / f'H{horizon}' / exp_name / 'ConvLSTM'
        pred_file = pred_dir / 'predictions.npy'
        targ_file = pred_dir / 'targets.npy'

        if pred_file.exists() and targ_file.exists():
            pred = np.load(pred_file)
            targ = np.load(targ_file)
            r2 = r2_score(targ.ravel(), pred.ravel())
            rmse = np.sqrt(np.mean((pred - targ)**2))
            print(f'\n{exp_name} (H{horizon}):')
            print(f'  predictions: {pred.shape}')
            print(f'  targets:     {targ.shape}')
            print(f'  R2={r2:.4f}, RMSE={rmse:.2f}mm')
        else:
            print(f'\n{exp_name} (H{horizon}): MISSING predictions!')

print('\n' + '=' * 60)
print('Next steps:')
print('1. Run V4 GNN-TAT notebook: train_v4_gnn_tat_intracell_dem.ipynb')
print('2. Then run V10 fusion: evaluate_v10_fusion_intracell_dem.ipynb')
print('3. Or locally: python workflows/run_pipeline.py --from 7 --intracell-dem')
print('=' * 60)